# Mobil Yüz Rekonstrüksiyon — Kaggle Eğitim Notebook'u

**GPU:** Kaggle P100/T4 (haftada 30 saat ücretsiz)  
**Çıktılar:** `/kaggle/working/` — session sonunda zip olarak indir  
**Kalıcı checkpoint:** Kaggle Dataset olarak kaydet (aşağıda açıklanıyor)

---
### Adımlar
1. Kurulum
2. Kodu GitHub'dan çek
3. Config (Kaggle yolları)
4. Veri seti hazırlama
5. Ön işleme
6. Eğitim
7. TFLite export
8. Checkpoint'i kalıcı kaydet

## 1. Kurulum

In [ ]:
import subprocess, sys

!nvidia-smi

# Eski onnxruntime kalintilarini temizle (CPU/GPU cakismasini onlemek icin)
!pip uninstall -y -q onnxruntime onnxruntime-gpu

# onnxruntime-gpu'yu SABIT bir surume (1.19.2) pinliyoruz — en yeni surum
# CUDA 13 runtime kutuphanesi (libcudart.so.13) istiyor ve Kaggle'da bu yok,
# cakip 'ImportError: libcudart.so.13 bulunamadi' hatasi veriyordu. 1.19.2,
# Kaggle'in kurulu CUDA 12.x runtime'i (torch +cu128) ile uyumlu.
# --no-deps: bu 3 paketin (onnxruntime-gpu, insightface, facenet-pytorch)
# transitive bagimliliklarinin numpy/protobuf gibi torch'un derlenmis C
# uzantilarinin ABI'sini bozabilecek surumlere otomatik guncellenmesini
# onler (bu, daha once torch'u 'SystemError: bad call flags' ile bozan
# --force-reinstall denemesinin sebebiydi).
!pip install -q --no-deps "onnxruntime-gpu==1.19.2" insightface facenet-pytorch pytorch-msssim

# Kalan paketler + protobuf'u google-cloud/tensorflow ile uyumlu bir surume sabitle
!pip install -q torchmetrics opencv-python-headless onnx tensorboard ai-edge-torch protobuf==5.26.1

print('Kurulum tamamlandı.')

import torch
print('torch:', torch.__version__, '| CUDA mevcut:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

import onnxruntime as ort
print('onnxruntime providers (insightface icin):', ort.get_available_providers())

## 2. Proje Kodunu GitHub'dan Çek

In [ ]:
import os, sys, shutil, importlib

# Idempotent: bu hücreyi ne zaman tekrar çalıştırırsan kod GitHub'dan
# yeniden çekilir. Kod güncellemesi lazım olduğunda SADECE bu hücreyi
# tekrar çalıştır, extraction/training hücrelerini elle düzenleme.
os.chdir('/kaggle/working')
shutil.rmtree('/kaggle/working/face-recon', ignore_errors=True)

!git clone https://github.com/BuhraOzdemir/face-recon.git /kaggle/working/face-recon

if '/kaggle/working/face-recon' not in sys.path:
    sys.path.insert(0, '/kaggle/working/face-recon')

os.chdir('/kaggle/working/face-recon')

# Bu session'da 'src.*' önceden import edildiyse eski (bellekteki) hali
# temizle — yeni kod diskten tekrar okunsun.
for mod_name in list(sys.modules):
    if mod_name == 'src' or mod_name.startswith('src.'):
        del sys.modules[mod_name]
importlib.invalidate_caches()

print('Proje kodu hazır (guncel):', os.getcwd())

## 3. Config (Kaggle Yolları)

Kaggle'da klasör yapısı:
```
/kaggle/input/DATASET_ADI/   ← eklediğin veri seti
/kaggle/working/             ← tüm çıktılar (checkpoint, export, log)
```

In [ ]:
from src.config import Config, DataConfig, ModelConfig, LossConfig, TrainConfig, ExportConfig

# Kaggle'a eklediğin veri setinin adını buraya yaz
# Notebook'un sag panelinde dataseti gordugunde URL'deki slug kismini al
# Ornek: kaggle.com/datasets/nguyncngtr/ms1mv3 -> 'ms1mv3'
DATASET_NAME = 'ms1mv3'  # <-- ekledığın datasetin Kaggle slug adı

cfg = Config(
    data=DataConfig(
        data_dir      = f'/kaggle/input/{DATASET_NAME}',
        processed_dir = '/kaggle/working/processed',
        image_size    = 128,
        val_split     = 0.05,
        num_workers   = 2,
    ),
    model=ModelConfig(
        embedding_dim    = 512,
        initial_spatial  = 4,
        initial_channels = 128,
        decoder_channels = (128, 128, 64, 32, 16),
    ),
    loss=LossConfig(
        phase1_epochs      = 3,   # warm-up (L1+perc, identity=0)
        phase2_epochs      = 10,  # ana eğitim (identity açık)
        phase3_epochs      = 7,   # ince ayar
        vgg_layer          = 'relu3_3',
        facenet_input_size = 160,
    ),
    train=TrainConfig(
        epochs            = 20,
        batch_size        = 64,
        learning_rate     = 1e-4,
        weight_decay      = 1e-4,
        warmup_epochs     = 2,
        save_dir          = '/kaggle/working/checkpoints',
        save_every_epochs = 5,
        keep_last_n       = 3,
        patience          = 10,
        use_amp           = True,
        log_dir           = '/kaggle/working/logs',
        log_every_steps   = 50,
    ),
    export=ExportConfig(
        export_dir  = '/kaggle/working/export',
        model_name  = 'face_decoder',
        quantize_int8 = True,
    ),
)

print(f'Toplam epoch : {cfg.total_epochs()}')
print(f'Veri giriş   : {cfg.data.data_dir}')
print(f'Checkpoint   : {cfg.train.save_dir}')
print(f'Export       : {cfg.export.export_dir}')

## 4. Veri Seti — Format Tespiti ve Çıkarım

MS1MV3 genellikle **InsightFace RecordIO** formatında gelir (`.rec` + `.idx` + `property`).  
Bu hücre formatı otomatik tespit eder ve RecordIO ise görüntüleri çıkarır.

> **Dataset slug'ını bulmak için:** Kaggle'da sağ panelde dataseti görünce 
> `▼ Input` altındaki yola bak: `/kaggle/input/SLUG_BURADA/`

In [ ]:
import os, struct
from pathlib import Path

data_path = Path(cfg.data.data_dir)

# Kaggle'daki tüm input dosyalarını listele
print('=== Kaggle Input Dosyaları ===')
for p in sorted(Path('/kaggle/input').rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f'  {str(p):<70}  {size_mb:>8.1f} MB')

print()

# Format tespiti
rec_files  = list(Path('/kaggle/input').rglob('*.rec'))
idx_files  = list(Path('/kaggle/input').rglob('*.idx'))
prop_files = list(Path('/kaggle/input').rglob('property'))

if rec_files:
    print('FORMAT: InsightFace RecordIO tespit edildi')
    print(f'  .rec : {rec_files[0]}')
    print(f'  .idx : {idx_files[0] if idx_files else "YOK"}')
    print(f'  prop : {prop_files[0] if prop_files else "YOK"}')
    DATA_FORMAT = 'recordio'
    REC_PATH    = str(rec_files[0])
    IDX_PATH    = str(idx_files[0]) if idx_files else None
    PROP_PATH   = str(prop_files[0]) if prop_files else None
else:
    id_dirs = [d for d in data_path.iterdir() if d.is_dir()] if data_path.exists() else []
    if id_dirs:
        print('FORMAT: Klasör yapısı (normal)')
        DATA_FORMAT = 'folder'
    else:
        print('UYARI: Veri bulunamadı — DATASET_NAME değişkenini kontrol et')
        DATA_FORMAT = 'none'

In [ ]:
# RecordIO formatını görüntü klasörlerine çıkar.
#
# NOT: Mantık artık src/data/extract_recordio.py içinde — bu hücre elle
# güncellenmez. Kod değişince yalnızca "2. Proje Kodunu GitHub'dan Çek"
# hücresini tekrar çalıştır (git clone), bu hücreyi DEĞİL.

EXTRACTED_DIR = '/kaggle/working/ms1mv3_images'

if DATA_FORMAT != 'recordio':
    print('RecordIO degil, bu hücre atlandı.')
else:
    from src.data.extract_recordio import extract_ms1mv3

    # max_per_id 15'e dusuruldu: Kaggle diski (~20GB) 30/id ile ~48K kimlikte
    # doluyordu (yalnizca ilk yarisi). 15/id ile TUM 93K kimlik sigar —
    # identity-loss tabanli egitim icin cesitlilik, kimlik basina goruntu
    # sayisindan daha degerli.
    result = extract_ms1mv3(
        rec_path=REC_PATH,
        extracted_dir=EXTRACTED_DIR,
        max_per_id=15,
        max_ids=93000,
    )
    cfg.data.data_dir = result['data_dir']

In [ ]:
from pathlib import Path

data_path = Path(cfg.data.data_dir)

if not data_path.exists():
    print(f'HATA: {data_path} bulunamadi!')
    print('extract_recordio hucresini calistirdin mi?')
else:
    id_dirs = [d for d in data_path.iterdir() if d.is_dir()]
    if not id_dirs:
        print(f'HATA: {data_path} bos — cikarim basarisiz olmus olabilir.')
        print('extract_recordio hucresini tekrar calistir.')
    else:
        sample = id_dirs[:200]
        total  = sum(len(list(d.glob('*.jpg'))) for d in sample)
        avg    = total // len(sample)
        print(f'Veri klasoru  : {data_path}')
        print(f'Kimlik sayisi : {len(id_dirs):,}')
        print(f'Ort. goruntu  : ~{avg} per kimlik')
        print(f'Toplam tahmin : ~{len(id_dirs) * avg:,}')
        print('Hazir!')

## 4b. IResNet50-ArcFace Ağırlıklarını İndir (Identity Supervisor)

Bu model **yalnızca eğitimde** kimlik denetleyicisi olarak kullanılır. Telefona yüklenmez.

InsightFace'in resmi MS1MV3-ArcFace R50 backbone dosyasını indir.
> Dosya ~87 MB, bir kez indirilir.

In [ ]:
import os, shutil, time, urllib.request

ARCFACE_PATH = '/kaggle/working/arcface_r50.pth'

if os.path.exists(ARCFACE_PATH) and os.path.getsize(ARCFACE_PATH) > 1_000_000:
    size_mb = os.path.getsize(ARCFACE_PATH) / 1e6
    print(f'IResNet50 ağırlıkları mevcut: {size_mb:.1f} MB')
else:
    # InsightFace resmi ms1mv3_arcface_r50 backbone .pth dosyası — Baidu/OneDrive'da
    # barındırılıyor (scriptli indirmeye kapalı). Hugging Face'te birebir aynı
    # dosyanın herkese açık aynası (mirror) mevcut — doğrudan indirilebiliyor.
    URL = 'https://huggingface.co/camenduru/show/resolve/main/models/arcface/ms1mv3_arcface_r50_fp16.pth'
    MAX_RETRIES = 4

    ARCFACE_PATH = None
    tmp_path = '/kaggle/working/arcface_r50.pth.tmp'
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f'Indiriliyor (deneme {attempt}/{MAX_RETRIES}): {URL}')
            # HF/CloudFront varsayilan Python User-Agent'ini 403 ile engelliyor —
            # tarayici benzeri bir User-Agent header ile istek gonder.
            req = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as resp, open(tmp_path, 'wb') as out_f:
                shutil.copyfileobj(resp, out_f)
            size_mb = os.path.getsize(tmp_path) / 1e6
            if size_mb < 50:
                raise RuntimeError(f'Dosya cok kucuk ({size_mb:.1f} MB) - indirme hatali olabilir')
            os.replace(tmp_path, '/kaggle/working/arcface_r50.pth')
            ARCFACE_PATH = '/kaggle/working/arcface_r50.pth'
            print(f'Basarili: {size_mb:.1f} MB kaydedildi -> {ARCFACE_PATH}')
            break
        except Exception as e:
            print(f'  Basarisiz: {e}')
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
            if attempt < MAX_RETRIES:
                wait_s = 5 * attempt   # 5s, 10s, 15s...
                print(f'  {wait_s}s sonra tekrar denenecek...')
                time.sleep(wait_s)

    if ARCFACE_PATH is None:
        print(f'{MAX_RETRIES} deneme basarisiz. FaceNet identity loss ile devam edilecek (yeterli ama optimal degil).')

# cfg zaten olusturulmus oldugu icin (Config hucresi bu hucreden once calisti)
# LossConfig'e arcface_r50_path'i burada elle isliyoruz.
cfg.loss.arcface_r50_path = ARCFACE_PATH
print(f'ARCFACE_PATH = {ARCFACE_PATH}')
print(f'cfg.loss.arcface_r50_path = {cfg.loss.arcface_r50_path}')

## 5. Ön İşleme

In [ ]:
# Daha önce işlediysen ve /kaggle/working/processed varsa atla
manifest_path = '/kaggle/working/processed/manifest.txt'

if os.path.exists(manifest_path):
    print(f'İşlenmiş veri zaten mevcut: {manifest_path}')
    with open(manifest_path) as f:
        n = sum(1 for l in f if l.strip())
    print(f'Örnek sayısı: {n:,}')
else:
    from src.data.preprocess import preprocess_dataset
    manifest_path = preprocess_dataset(
        raw_dir     = cfg.data.data_dir,
        out_dir     = cfg.data.processed_dir,
        output_size = cfg.data.image_size,
        max_per_id  = 100,
    )

In [ ]:
from src.data.dataset import build_dataloaders
import torch

# NOT: split KIMLIK BAZLI — bir kimligin tum goruntuleri sadece TEK bir
# split'te bulunur (leakage yok). test_loader egitimde kullanilmaz, sadece
# egitim bittikten sonra tek seferlik nihai degerlendirme icindir.
train_loader, val_loader, test_loader = build_dataloaders(
    manifest_path = manifest_path,
    image_size    = cfg.data.image_size,
    batch_size    = cfg.train.batch_size,
    val_split     = cfg.data.val_split,
    test_split    = cfg.data.test_split,
    num_workers   = cfg.data.num_workers,
)
print(f'Train: {len(train_loader.dataset):,}  |  Val: {len(val_loader.dataset):,}  |  Test: {len(test_loader.dataset):,}')

emb, img = next(iter(train_loader))
print(f'Embedding: {emb.shape}  |  Görüntü: {img.shape}  |  Aralık: [{img.min():.2f}, {img.max():.2f}]')

## 5b. Faz A — 32 Örnek Overfit Sanity Check

Ana eğitime (tam veri seti, çok epoch) geçmeden önce, pipeline'ın
gerçekten öğrenebildiğini ucuza doğruluyoruz. Küçük bir örnek kümesi
üzerinde saf L1 loss ile birkaç yüz adım eğitip **ezberleyebiliyor mu**
diye bakıyoruz.

- **Final L1 < 0.03 VE görüntüler görsel olarak yakınsıyorsa** → mimari/pipeline sağlam, ana eğitimdeki sorun muhtemelen "yetersiz epoch/veri".
- **Loss düşmüyorsa veya görüntüler hâlâ anlamsızsa (renk blob'u vb.)** → gerçek bir bug var, ana eğitime geçmeden bunu çöz.

In [ ]:
from src.overfit_check import run_overfit_test

final_l1 = run_overfit_test(
    cfg            = cfg,
    manifest_path  = manifest_path,
    n_samples      = 32,
    steps          = 800,
)


## 6. Eğitim

In [ ]:
from src.train import train

# Önceki session'dan checkpoint varsa devam et
resume_path = '/kaggle/working/checkpoints/best_model.pt'
if not os.path.exists(resume_path):
    resume_path = None

if resume_path:
    print(f'Checkpoint bulundu, devam ediliyor: {resume_path}')
else:
    print('Sıfırdan eğitim başlıyor...')

best_ckpt = train(
    cfg           = cfg,
    manifest_path = manifest_path,
    resume_from   = resume_path,
)
print(f'Eğitim tamamlandı: {best_ckpt}')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/logs

## 7. Sonuçları Görselleştir

In [ ]:
import torch, matplotlib.pyplot as plt
from src.models.decoder import FaceDecoder
from src.data.dataset import denormalize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = FaceDecoder(
    embedding_dim    = cfg.model.embedding_dim,
    initial_spatial  = cfg.model.initial_spatial,
    initial_channels = cfg.model.initial_channels,
    decoder_channels = cfg.model.decoder_channels,
).to(device)

state = torch.load(best_ckpt, map_location=device)
model.load_state_dict(state['model'])
model.eval()

val_embs, val_imgs = next(iter(val_loader))
with torch.no_grad():
    generated = model(val_embs[:8].to(device)).cpu()

real_np = denormalize(val_imgs[:8]).permute(0,2,3,1).numpy()
gen_np  = denormalize(generated).permute(0,2,3,1).numpy()

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i in range(8):
    axes[0,i].imshow(real_np[i].clip(0,1)); axes[0,i].set_title('Gerçek', fontsize=8); axes[0,i].axis('off')
    axes[1,i].imshow(gen_np[i].clip(0,1));  axes[1,i].set_title('Üretilen', fontsize=8); axes[1,i].axis('off')
plt.suptitle('Gerçek vs Üretilen', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/results_sample.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. TFLite Export

In [ ]:
from src.export_tflite import export_model

exported = export_model(
    checkpoint_path = best_ckpt,
    export_dir      = cfg.export.export_dir,
    cfg             = cfg,
)

print('\nExport tamamlandı:')
for fmt, path in exported.items():
    size_mb = os.path.getsize(path) / 1e6
    print(f'  {fmt:12s}: {size_mb:.2f} MB  →  {path}')

## 9. Checkpoint'i Kalıcı Kaydet

Kaggle session kapandığında `/kaggle/working/` silinir.  
Checkpoint'i **Kaggle Dataset** olarak kaydetmek için:

In [ ]:
# Önemli dosyaları tek zip'e topla
import zipfile, shutil

zip_path = '/kaggle/working/face_recon_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Best model checkpoint
    if os.path.exists('/kaggle/working/checkpoints/best_model.pt'):
        zf.write('/kaggle/working/checkpoints/best_model.pt', 'best_model.pt')
    # TFLite dosyaları
    for fmt, path in exported.items():
        if os.path.exists(path):
            zf.write(path, os.path.basename(path))
    # Örnek görüntüler
    if os.path.exists('/kaggle/working/results_sample.png'):
        zf.write('/kaggle/working/results_sample.png', 'results_sample.png')

size_mb = os.path.getsize(zip_path) / 1e6
print(f'Zip oluşturuldu: {zip_path}  ({size_mb:.1f} MB)')
print()
print('Kalıcı kaydetmek için:')
print('  1. Sağ panelde Output sekmesine tıkla')
print('  2. face_recon_results.zip yanındaki üç noktaya tıkla')
print('  3. "Save to Dataset" seç → yeni dataset oluştur')
print('  4. Sonraki session\'da bu dataseti notebook\'a ekle')

In [ ]:
# Sonraki session'da checkpoint'i geri yüklemek için:
# 1. Kaggle dataset'ini notebook'a ekle
# 2. Aşağıdaki kodu çalıştır:

# import shutil, zipfile
# with zipfile.ZipFile('/kaggle/input/DATASET_ADIN/face_recon_results.zip') as zf:
#     zf.extract('best_model.pt', '/kaggle/working/checkpoints/')
# print('Checkpoint geri yüklendi.')